# Coordination indices by département — presidential first rounds 2002 & 2022

Computes, for each `year × département`, the coordination measures of the first-round result from the
candidates' vote shares among expressed votes, $\delta = (\delta_1, \dots, \delta_K)$:

| measure | definition |
|---|---|
| **ENP** | $1 / \sum_j \delta_j^2$ |
| **CENP** (main measure) | $(K - \text{ENP}) / (K - 1)$ — 0 = uniform across the $K$ candidates, 1 = all votes on one candidate |
| **cliff magnitude** | $d^* = \max_k g_k$ with $g_k = \delta_{(k)} - \delta_{(k+1)}$, shares sorted in decreasing order |
| **cliff location** | $\arg\max_k g_k$, 1-based: `2` = largest drop between the 2nd and 3rd candidates |
| **cliff ratio** | $d^* / (d^* + \overline{g}_{-})$, with $\overline{g}_{-}$ the mean of the other consecutive gaps |

CENP describes the observed result only (there is no département-level sincere counterfactual here),
so it is not a coordination *gain*.

`K` is the number of candidates in the election (16 in 2002, 12 in 2022), identical for all départements
of a given year; a candidate with no vote in a département enters with $\delta_j = 0$.

**Input:** `data/processed/presidential_departments_2002_2022.csv`, produced by
`departements_results_scrapping.ipynb` (long format, one row per `year × département × candidate`).

## 1. Imports

In [16]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

## 2. Load département-level results

In [17]:
INPUT_PATH = Path("data/processed/presidential_departments_2002_2022.csv")
OUTPUT_PATH = Path("data/processed/coordination_indices_departements.csv")

# Number of first-round candidates in each election
EXPECTED_K = {2002: 16, 2022: 12}

# dep_code must stay text: '01', '2A', '2B', '971', 'ZZ', ...
results = pd.read_csv(INPUT_PATH, dtype={"dep_code": str}, encoding="utf-8")

print(results.shape)
print(results.groupby("year").agg(departements=("dep_code", "nunique"),
                                  candidates=("candidate_clean", "nunique"),
                                  rows=("votes", "size")))
results.head()

(2976, 16)
      departements  candidates  rows
year                                
2002           105          16  1680
2022           108          12  1296


,year,round,dep_code,dep_name,dep_type,inscrits,abstentions,votants,blancs_nuls,exprimes,candidate,candidate_clean,candidate_surname,votes,vote_share,vote_share_calc
0,2002,1,01,Ain,metropole,338220,89002,249218,8566,240652,M. JEAN-MARIE LE PEN,JEAN-MARIE LE PEN,LE PEN,52617,21.86,21.864352
1,2002,1,01,Ain,metropole,338220,89002,249218,8566,240652,M. JACQUES CHIRAC,JACQUES CHIRAC,CHIRAC,41348,17.18,17.181656
2,2002,1,01,Ain,metropole,338220,89002,249218,8566,240652,M. LIONEL JOSPIN,LIONEL JOSPIN,JOSPIN,30418,12.64,12.639828
3,2002,1,01,Ain,metropole,338220,89002,249218,8566,240652,M. FRANCOIS BAYROU,FRANCOIS BAYROU,BAYROU,18614,7.73,7.734820
4,2002,1,01,Ain,metropole,338220,89002,249218,8566,240652,M. JEAN-PIERRE CHEVENEMENT,JEAN-PIERRE CHEVENEMENT,CHEVENEMENT,14665,6.09,6.093862


### Entries without a name

In 2022 the scraper found Saint-Martin/Saint-Barthélemy twice: once as `ZX` (from its link label) and
once as `977` (from its URL), with the same page and no name for `977`. Rows without a département name
are such duplicates and are dropped here.

In [18]:
unnamed = results[results["dep_name"].isna()]
display(unnamed.drop_duplicates(["year", "dep_code"])[["year", "dep_code", "dep_name", "dep_type"]])

results = results[results["dep_name"].notna()].copy()
print(results.groupby("year")["dep_code"].nunique())

,year,dep_code,dep_name,dep_type
2904,2022,977,NaN,NaN


year
2002    105
2022    107
Name: dep_code, dtype: int64


## 3. Clean / check vote shares

The shares used for the indices are **recomputed from the counts**:

$$\delta_j = \frac{\text{votes}_j}{\sum_k \text{votes}_k}$$

The denominator is the sum of the candidates' votes, i.e. the first-round expressed votes. The published
`vote_share` column (0–100 scale, rounded to 2 decimals) is only used as a check.

> **Warning — turnout columns.** Check below whether the scraped `exprimes` equals the sum of candidate
> votes. On the 2022 pages the turnout figures (`inscrits`, `votants`, `exprimes`, …) do not match the
> first-round candidate votes in most départements (they seem to come from the second round), while the
> candidate votes and published shares are consistent with each other. The indices do not depend on
> these columns; the turnout columns are only reported in the output where they are consistent.

In [19]:
by_dep = results.groupby(["year", "dep_code"]).agg(
    sum_votes=("votes", "sum"),
    exprimes=("exprimes", "first"),
    sum_published_share=("vote_share", "sum"),
).reset_index()
by_dep["turnout_consistent"] = by_dep["sum_votes"] == by_dep["exprimes"]

print("Départements where scraped 'exprimes' == sum of candidate votes:")
print(by_dep.groupby("year")["turnout_consistent"].agg(["sum", "size"]))

Départements where scraped 'exprimes' == sum of candidate votes:
      sum  size
year           
2002  105   105
2022  107   107


In [20]:
# Recomputed shares (proportions, 0–1)
results["share"] = results["votes"] / results.groupby(["year", "dep_code"])["votes"].transform("sum")

# Published shares are on a 0–100 scale: convert before comparing
published = results["vote_share"] / 100

share_sums = results.groupby(["year", "dep_code"])["share"].sum()
print("Recomputed shares sum to 1 in every département:", np.allclose(share_sums, 1))

print("Published shares (0–100) sum to ~100 (±0.1):",
      ((by_dep["sum_published_share"] - 100).abs() <= 0.1).all())

gap = (results["share"] - published).abs()
print("Max |recomputed − published| share:", round(gap.max(), 6), "(rounding to 2 decimals in % gives ≤ 0.00005)")
results.loc[gap > 0.0001, ["year", "dep_code", "candidate_clean", "votes", "vote_share", "share"]]

Recomputed shares sum to 1 in every département: True
Published shares (0–100) sum to ~100 (±0.1): True
Max |recomputed − published| share: 5e-05 (rounding to 2 decimals in % gives ≤ 0.00005)


,year,dep_code,candidate_clean,votes,vote_share,share


### Candidate × département grid

`K` is the number of candidates in the election. Every département gets all `K` candidates
(a candidate absent from a département's results would get a share of 0), so all départements of a
given year use the same `K`.

In [21]:
K_BY_YEAR = results.groupby("year")["candidate_clean"].nunique().to_dict()
print("K by year:", K_BY_YEAR, "| expected:", EXPECTED_K)
assert K_BY_YEAR == EXPECTED_K, "number of candidates differs from the expected first-round field"


def shares_matrix(results, year):
    """Départements × candidates matrix of shares for one year (0 for a candidate with no row)."""
    sub = results[results["year"] == year]
    return sub.pivot_table(index="dep_code", columns="candidate_clean", values="share",
                           aggfunc="sum", fill_value=0.0)


for year in EXPECTED_K:
    m = shares_matrix(results, year)
    print(f"{year}: {m.shape[0]} départements × {m.shape[1]} candidates; "
          f"cells filled with 0: {(m == 0).sum().sum()}")

K by year: {2002: 16, 2022: 12} | expected: {2002: 16, 2022: 12}
2002: 105 départements × 16 candidates; cells filled with 0: 0
2022: 107 départements × 12 candidates; cells filled with 0: 0


## 4. Coordination-measure functions

In [22]:
def compute_enp(shares):
    """Effective number of parties: 1 / sum(delta_j^2), shares as proportions."""
    shares = np.asarray(shares, dtype=float)
    return 1.0 / np.sum(shares ** 2)


def compute_cenp(shares, K):
    """CENP = (K - ENP) / (K - 1): 0 = uniform over the K candidates, 1 = one candidate gets everything.

    `shares` must contain all K candidates of the election (zeros included).
    """
    shares = np.asarray(shares, dtype=float)
    assert len(shares) == K, f"expected {K} shares, got {len(shares)}"
    return (K - compute_enp(shares)) / (K - 1)


def compute_cliff_metrics(shares):
    """Largest drop between consecutive ranked shares.

    Shares are sorted in decreasing order, gaps g_k = delta_(k) - delta_(k+1), k = 1..K-1.
    - cliff_magnitude: d* = max_k g_k
    - cliff_location: argmax_k g_k, 1-based (2 = between the 2nd and 3rd candidates);
      if several gaps are equal, the first (highest-ranked) one is taken
    - cliff_ratio: d* / (d* + mean of the other gaps), in [0.5, 1]; NaN if all gaps are 0
    """
    ranked = np.sort(np.asarray(shares, dtype=float))[::-1]
    gaps = ranked[:-1] - ranked[1:]
    k = int(np.argmax(gaps))
    d_star = gaps[k]
    other_gaps = np.delete(gaps, k)
    mean_other_gaps = other_gaps.mean() if len(other_gaps) else 0.0
    denom = d_star + mean_other_gaps
    return {
        "cliff_magnitude": d_star,
        "cliff_location": k + 1,
        "cliff_ratio": d_star / denom if denom > 0 else np.nan,
    }

Quick checks of the functions on toy vectors:

In [23]:
uniform = np.full(4, 0.25)
concentrated = np.array([1.0, 0.0, 0.0, 0.0])
example = np.array([0.40, 0.35, 0.15, 0.10])  # gaps: 0.05, 0.20, 0.05 -> cliff between 2nd and 3rd

print("uniform:      ENP", compute_enp(uniform), "| CENP", compute_cenp(uniform, 4))
print("concentrated: ENP", compute_enp(concentrated), "| CENP", compute_cenp(concentrated, 4))
print("example:      ", compute_cliff_metrics(example), "| expected ratio", 0.20 / (0.20 + 0.05))

assert np.isclose(compute_cenp(uniform, 4), 0) and np.isclose(compute_cenp(concentrated, 4), 1)
assert compute_cliff_metrics(example)["cliff_location"] == 2
assert np.isclose(compute_cliff_metrics(example)["cliff_ratio"], 0.8)

uniform:      ENP 4.0 | CENP 0.0
concentrated: ENP 1.0 | CENP 1.0
example:       {'cliff_magnitude': 0.19999999999999998, 'cliff_location': 2, 'cliff_ratio': 0.7999999999999999} | expected ratio 0.8


## 5. Compute indices by département

In [24]:
def compute_department_indices(results, K_by_year):
    """One row per year × département with the coordination indices and context variables."""
    rows = []
    for year, K in K_by_year.items():
        matrix = shares_matrix(results, year)
        assert matrix.shape[1] == K
        for dep_code, shares in matrix.iterrows():
            rows.append({
                "year": year,
                "department_code": dep_code,
                "K": K,
                "ENP": compute_enp(shares.values),
                "CENP": compute_cenp(shares.values, K),
                **compute_cliff_metrics(shares.values),
                "winner": shares.idxmax(),
                "winner_vote_share": shares.max(),
            })
    return pd.DataFrame(rows)


indices = compute_department_indices(results, K_BY_YEAR)

# Context variables: names, type, expressed votes (= sum of candidate votes) and turnout where consistent
context = (results.groupby(["year", "dep_code"])
                  .agg(department_name=("dep_name", "first"),
                       department_type=("dep_type", "first"),
                       registered=("inscrits", "first"),
                       voters=("votants", "first"),
                       expressed=("votes", "sum"))
                  .reset_index()
                  .rename(columns={"dep_code": "department_code"})
                  .merge(by_dep[["year", "dep_code", "turnout_consistent"]]
                         .rename(columns={"dep_code": "department_code"}),
                         on=["year", "department_code"]))
# Registered/voters figures that do not match the first-round votes are not reported
context.loc[~context["turnout_consistent"], ["registered", "voters"]] = np.nan

coordination = indices.merge(context, on=["year", "department_code"], how="left")
coordination = coordination[[
    "year", "department_code", "department_name", "department_type", "K",
    "ENP", "CENP", "cliff_magnitude", "cliff_location", "cliff_ratio",
    "winner", "winner_vote_share", "registered", "voters", "expressed", "turnout_consistent",
]].sort_values(["year", "department_code"]).reset_index(drop=True)

coordination.head(10)

,year,department_code,department_name,department_type,K,ENP,CENP,cliff_magnitude,cliff_location,cliff_ratio,winner,winner_vote_share,registered,voters,expressed,turnout_consistent
0,2002,01,Ain,metropole,16,8.582938,0.494471,0.049050,3,0.806158,JEAN-MARIE LE PEN,0.218644,338220.0,249218.0,240652,True
1,2002,02,Aisne,metropole,16,8.019706,0.532020,0.071963,3,0.883307,JEAN-MARIE LE PEN,0.212228,366810.0,270653.0,261803,True
2,2002,03,Allier,metropole,16,8.745732,0.483618,0.059916,3,0.853955,JACQUES CHIRAC,0.208786,256113.0,189047.0,179768,True
3,2002,04,Alpes-de-Haute-Provence,metropole,16,9.834442,0.411037,0.046859,3,0.852148,JEAN-MARIE LE PEN,0.165983,108943.0,82388.0,79430,True
4,2002,05,Hautes-Alpes,metropole,16,10.251129,0.383258,0.058918,3,0.882432,JACQUES CHIRAC,0.173424,95100.0,72049.0,69402,True
5,2002,06,Alpes-Maritimes,metropole,16,6.801695,0.613220,0.097893,2,0.895987,JEAN-MARIE LE PEN,0.259854,690348.0,478074.0,467040,True
6,2002,07,Ardèche,metropole,16,9.574919,0.428339,0.065378,3,0.899906,JACQUES CHIRAC,0.172573,219920.0,167668.0,161445,True
7,2002,08,Ardennes,metropole,16,7.720407,0.551973,0.088983,3,0.902049,JEAN-MARIE LE PEN,0.229193,192422.0,138608.0,134498,True
8,2002,09,Ariège,metropole,16,8.100288,0.526647,0.086212,1,0.891630,LIONEL JOSPIN,0.237294,109027.0,82875.0,79606,True
9,2002,10,Aube,metropole,16,7.802254,0.546516,0.074374,3,0.883313,JEAN-MARIE LE PEN,0.217236,193749.0,142924.0,138140,True


## 6. Validation

In [25]:
def report(label, ok, detail=None):
    print(f"[{'OK' if ok else 'CHECK'}] {label}")
    if not ok and detail is not None and len(detail):
        display(detail)


INDEX_COLUMNS = ["ENP", "CENP", "cliff_magnitude", "cliff_location", "cliff_ratio"]

### Number of départements, duplicates, missing values

In [26]:
print("Rows per year:")
display(coordination.groupby(["year", "department_type"]).size().unstack(fill_value=0).assign(total=lambda d: d.sum(axis=1)))

dups = coordination.duplicated(["year", "department_code"], keep=False)
report("no duplicated year × department_code rows", not dups.any(), coordination[dups])

missing = coordination[INDEX_COLUMNS].isna().any(axis=1)
report("no missing value in any index", not missing.any(), coordination[missing])

report("Corsica split into 2A and 2B in both years",
       all({"2A", "2B"} <= set(coordination.loc[coordination["year"] == y, "department_code"]) for y in EXPECTED_K))

Rows per year:


department_type,COM,DOM,etranger,metropole,total
year,,,,,
2002,4,5,0,96,105
2022,5,5,1,96,107


[OK] no duplicated year × department_code rows
[OK] no missing value in any index
[OK] Corsica split into 2A and 2B in both years


### Summary of the indices by year

In [27]:
coordination.groupby("year")[INDEX_COLUMNS].agg(["min", "max", "mean"]).T

year                       2002      2022
ENP             min    2.208966  2.718799
                max   10.251129  6.210927
                mean   7.980329  5.021627
CENP            min    0.383258  0.526279
                max    0.919402  0.843746
                mean   0.534645  0.634398
cliff_magnitude min    0.040178  0.078434
                max    0.376699  0.382411
                mean   0.094025  0.135780
cliff_location  min    1.000000  1.000000
                max    4.000000  4.000000
                mean   2.371429  2.420561
cliff_ratio     min    0.783155  0.800437
                max    0.988447  0.959100
                mean   0.890423  0.880625

### Range checks

In [28]:
for year, K in EXPECTED_K.items():
    sub = coordination[coordination["year"] == year]
    report(f"{year}: K == {K} for every département", sub["K"].eq(K).all(), sub.loc[sub["K"].ne(K), ["department_code", "K"]])

report("CENP between 0 and 1", coordination["CENP"].between(0, 1).all(),
       coordination.loc[~coordination["CENP"].between(0, 1), ["year", "department_code", "CENP"]])

report("ENP between 1 and K", (coordination["ENP"].ge(1 - 1e-12) & coordination["ENP"].le(coordination["K"] + 1e-12)).all())

# d* >= every other gap >= their mean, so the ratio is >= 0.5; it is 1 only if all other gaps are 0
report("cliff ratio between 0.5 and 1", coordination["cliff_ratio"].between(0.5, 1).all(),
       coordination.loc[~coordination["cliff_ratio"].between(0.5, 1), ["year", "department_code", "cliff_ratio"]])

loc_ok = coordination["cliff_location"].between(1, coordination["K"] - 1)
report("cliff location between 1 and K − 1", loc_ok.all(),
       coordination.loc[~loc_ok, ["year", "department_code", "K", "cliff_location"]])

report("cliff magnitude between 0 and 1", coordination["cliff_magnitude"].between(0, 1).all())

print("\nDistribution of cliff locations:")
display(coordination.groupby("year")["cliff_location"].value_counts().unstack(fill_value=0))

[OK] 2002: K == 16 for every département
[OK] 2022: K == 12 for every département
[OK] CENP between 0 and 1
[OK] ENP between 1 and K
[OK] cliff ratio between 0.5 and 1
[OK] cliff location between 1 and K − 1
[OK] cliff magnitude between 0 and 1

Distribution of cliff locations:


cliff_location,1,2,3,4
year,,,,
2002,22,23,59,1
2022,24,15,67,1


## 7. Inspect a few examples

Ranked vote shares (in %) for a few départements, with the gaps and the resulting cliff, to check the
calculation by hand. The cliff gap is marked with `◀`.

In [29]:
def ranked_shares_table(results, year, dep_codes):
    """One row per rank: share of the k-th candidate, gap to the next one, for each département."""
    matrix = shares_matrix(results, year)
    out = []
    for dep_code in dep_codes:
        ranked = matrix.loc[dep_code].sort_values(ascending=False)
        gaps = ranked.values[:-1] - ranked.values[1:]
        cliff = compute_cliff_metrics(ranked.values)
        for rank, (candidate, share) in enumerate(ranked.items(), 1):
            gap = gaps[rank - 1] if rank < len(ranked) else np.nan
            out.append({
                "department_code": dep_code,
                "rank": rank,
                "candidate": candidate,
                "share_%": round(100 * share, 2),
                "gap_to_next_pp": round(100 * gap, 2),
                "cliff": "◀" if rank == cliff["cliff_location"] else "",
            })
    return pd.DataFrame(out)


EXAMPLES = ["75", "2A", "971"]

for year in EXPECTED_K:
    print(f"\n===== {year} =====")
    display(coordination.loc[(coordination["year"] == year) & coordination["department_code"].isin(EXAMPLES),
                             ["department_code", "department_name", "ENP", "CENP",
                              "cliff_magnitude", "cliff_location", "cliff_ratio"]])
    table = ranked_shares_table(results, year, EXAMPLES)
    display(table.pivot(index="rank", columns="department_code", values=["share_%", "gap_to_next_pp", "cliff"]))


===== 2002 =====


,department_code,department_name,ENP,CENP,cliff_magnitude,cliff_location,cliff_ratio
28,2A,Corse-du-Sud,6.770516,0.615299,0.102465,1,0.893643
75,75,Paris,7.605376,0.559642,0.105930,2,0.918932
96,971,Guadeloupe,3.609502,0.826033,0.201845,3,0.943399


share_%               gap_to_next_pp               cliff       
department_code      2A     75    971             2A     75    971    2A 75 971
rank                                                                           
1                 27.65  24.01  37.22          10.25   4.07   8.33     ◀       
2                 17.41  19.94   28.9           2.41  10.59   5.77        ◀    
3                  15.0   9.35  23.12           7.27   1.44  20.18            ◀
4                  7.73   7.91   2.94           0.96   0.52   1.52             
5                  6.76   7.39   1.42           2.07   0.76   0.35             
6                   4.7   6.62   1.06           1.05   0.12   0.17             
7                  3.64    6.5   0.89           0.33   2.72   0.01             
8                  3.31   3.78   0.88           0.07   0.36   0.12             
9                  3.24   3.42   0.76           0.44   0.37    0.1             
10                  2.8   3.05   0.66           0.42   0.59    0.1             
11                 2.38   2.46   0.55           0.07   0.27   0.03             
12                 2.31   2.19   0.53           0.88   0.69   0.17             
13                 1.43   1.49   0.36            0.6   0.47   0.03             
14                 0.84   1.02   0.33           0.39    0.5   0.02             
15                 0.45   0.52   0.31           0.11   0.19   0.22             
16                 0.33   0.34   0.08            NaN    NaN    NaN


===== 2022 =====


,department_code,department_name,ENP,CENP,cliff_magnitude,cliff_location,cliff_ratio
133,2A,Corse-du-Sud,5.819453,0.561868,0.104660,1,0.849832
180,75,Paris,4.233657,0.706031,0.219214,2,0.943444
201,971,Guadeloupe,2.718799,0.843746,0.382411,1,0.956424


share_%               gap_to_next_pp               cliff       
department_code      2A     75    971             2A     75    971    2A 75 971
rank                                                                           
1                 29.25  35.34  56.16          10.47   5.26  38.24     ◀      ◀
2                 18.78  30.08  17.92           5.23  21.92   4.49        ◀    
3                 13.55   8.16  13.43           0.19   0.55  10.48             
4                 13.36   7.61   2.95           3.59   1.01   0.65             
5                  9.77   6.59   2.29           4.29   1.06   0.62             
6                  5.48   5.54   1.68           2.19   3.37   0.11             
7                  3.29   2.17   1.56            0.6   0.53   0.14             
8                  2.69   1.64   1.43           0.92   0.49   0.62             
9                  1.76   1.15    0.8           0.86   0.24   0.04             
10                 0.91   0.91   0.76           0.02   0.37   0.24             
11                 0.88   0.54   0.53            0.6   0.27   0.03             
12                 0.29   0.27   0.49            NaN    NaN    NaN

## 8. Save output

UTF-8 CSV, one row per `year × département`. When reading it back keep the codes as text:
`pd.read_csv(OUTPUT_PATH, dtype={"department_code": str})`.

In [30]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
coordination.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
print(f"Saved {len(coordination)} rows to {OUTPUT_PATH}")

check = pd.read_csv(OUTPUT_PATH, dtype={"department_code": str}, encoding="utf-8")
check[check["department_code"].isin(["01", "2A", "2B", "971"])][["year", "department_code", "department_name", "CENP"]]

Saved 212 rows to data/processed/coordination_indices_departements.csv


,year,department_code,department_name,CENP
0,2002,01,Ain,0.494471
28,2002,2A,Corse-du-Sud,0.615299
29,2002,2B,Haute-Corse,0.598854
96,2002,971,Guadeloupe,0.826033
105,2022,01,Ain,0.609986
133,2022,2A,Corse-du-Sud,0.561868
134,2022,2B,Haute-Corse,0.528422
201,2022,971,Guadeloupe,0.843746
